# Same Story, Different Headlines

In this notebook, you'll compare how four different news outlets, the
New York Times, the Wall Street Journal, Fox News, and China's state agency
Xinhua, headline the "same" US-China events. Then you'll train a model to guess
which outlet wrote a headline from its wording alone.

Almost everything is pre-written, and in the marked spots
you'll edit lines code and see what changes.

---

### Step 1

Watch the intro video:

This notebook is your personal copy and your edits save
automatically. If it ever gets hopelessly tangled, **Ctrl+Z / Cmd+Z** inside a cell
fixes almost everything but feel free to message/email Heagen for a new one or any questions if needed.

### How it works

- Cells run top to bottom: click a cell, press **Shift+Enter** (or ▶).
- **JUST RUN** cells: run, don't edit.
- **EDIT** cells: change the line marked `# EDIT`.

## Setup

**JUST RUN**: installs two libraries and loads the headlines. Note that this may take a bit.

In [ ]:
%pip install -q sentence-transformers vaderSentiment

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_colwidth", 100)

DATA_URL = "https://raw.githubusercontent.com/heagenb03/twl-hs-interns-notebooks/main/data/headlines_real.csv"
headlines = pd.read_csv(DATA_URL)
print(f"Loaded {len(headlines)} headlines: {headlines.outlet.nunique()} outlets x "
      f"{headlines.event.nunique()} events.")
headlines.head()

> **What you should see:** `Loaded 72 headlines: 4 outlets x 6 events.` and a
> table with columns `event`, `outlet`, `headline`.

Which events are covered? **JUST RUN:**

In [ ]:
# JUST RUN: the six events
list(headlines["event"].unique())

> **What you should see:** six event names, from `'2018 tariff announcement'` to
> `'Spy balloon'`.

**JUST RUN**: Example outlets to headlines

In [ ]:
# JUST RUN
event_to_read = list(headlines["event"].unique())[2]
headlines[headlines.event == event_to_read][["outlet", "headline"]]

> **What you should see:** A table (i.e. a dataframe) with the outlet corresponding to their headline. Can you tell the difference? Click the calculator looking icon to see the full table.

We can conceptually tell the difference but how could a computer tell or "measure" the difference? **JUST RUN:**

In [ ]:
# JUST RUN
vader = SentimentIntensityAnalyzer()
headlines["sentiment"] = headlines["headline"].map(
    lambda t: vader.polarity_scores(t)["compound"])

def sentiment_by_outlet(event):
    """Average headline sentiment per outlet for one event. -1 = very negative, +1 = very positive."""
    d = headlines[headlines.event == event]
    if len(d) == 0:
        print(f"(No event called '{event}'. The options: {list(headlines.event.unique())})")
        return
    means = d.groupby("outlet")["sentiment"].mean().reindex(["NYT", "WSJ", "Fox", "Xinhua"])
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ["#2ca02c" if v >= 0 else "#d62728" for v in means]
    ax.bar(means.index, means.values, color=colors)
    ax.axhline(0, color="gray")
    ax.set_ylim(-1, 1); ax.set_ylabel("avg. sentiment score")
    ax.set_title(f'Headline sentiment: "{event}"')
    plt.tight_layout(); plt.show()
    for outlet, v in means.items():
        print(f"{outlet}: {v:+.2f}")

def score_one(text):
    """VADER sentiment of any single sentence you type."""
    s = vader.polarity_scores(text)["compound"]
    print(f'"{text}"  →  sentiment {s:+.2f}')

print("Sentiment tools ready")

---
## Measuring the spin

We'll score every headline with a **sentiment analyzer** (called VADER), a tool
that rates text from −1 (very negative) to +1 (very positive) by adding up scores
of individual words.

**EDIT** run as-is first, then swap in other events from the list above.

In [ ]:
event = "Hong Kong security law"   # <-- EDIT (copy any event name from the list above)

sentiment_by_outlet(event)

> **What you should see** (with the starting event): a bar chart and printed scores
> roughly `NYT: -0.13  WSJ: -0.06  Fox: -0.03  Xinhua: +0.17`.

### Things to try

1. Look at that Phase One pattern. The two outlets closest to the two *governments*
   that signed the deal (Fox → Trump administration, Xinhua → Beijing) both spin
   near/are positive. The other two papers of record are both skeptical. Why do you think that is? Scroll up to the raw headlines and try to find the exact
   words driving each score.
2. Try `"Spy balloon"`. Unfortunately could not get data on Xinhua but what would you think it's score would be?

---
## Guess the outlet

Can a model tell who wrote the headline from
wording alone?

We'll use a type of language-based model that turns each headline into numbers in attempt to capture its
"meaning and style", then train a classifier to map those numbers to outlets. We will use 22 headlines that the model never sees to test it.

**JUST RUN** (takes ~1 minute, since it's downloading the language model):

In [ ]:
# JUST RUN
embedder = SentenceTransformer("all-MiniLM-L6-v2")
X = embedder.encode(list(headlines["headline"]), show_progress_bar=True)
y = headlines["outlet"].values

X_train, X_test, y_train, y_test, head_train, head_test = train_test_split(
    X, y, headlines, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Trained on {len(X_train)} headlines.")
print(f"On {len(X_test)} headlines it has NEVER seen, it guesses the outlet right {accuracy:.0%} of the time.")
print(f"(Random guessing among 4 outlets would get 25%.)")

def guess_outlet(text):
    """The model's guess with its confidence in each of the four outlets."""
    probs = model.predict_proba(embedder.encode([text]))[0]
    order = np.argsort(probs)[::-1]
    print(f'"{text}"')
    for i in order:
        bar = "█" * int(probs[i] * 30)
        print(f"  {model.classes_[i]:<7} {probs[i]:>5.0%} {bar}")

def quiz():
    """Show a random test headline. Guess the outlet yourself, then reveal."""
    row = head_test.sample(1).iloc[0]
    guess = model.predict(embedder.encode([row.headline]))[0]
    print(f'HEADLINE: "{row.headline}"')
    print(f"Model's guess: {guess}   |   Actual: {row.outlet}   |   "
          f"{'model was RIGHT' if guess == row.outlet else 'model was WRONG'}")

def battery(my_headlines):
    """Run a list of your own headlines through the classifier."""
    if isinstance(my_headlines, str):
        my_headlines = [my_headlines]
    rows = [(h[:65], model.predict(embedder.encode([h]))[0]) for h in my_headlines]
    return pd.DataFrame(rows, columns=["your headline", "model's guess"])

print("Done")

> **What you should see:** an accuracy line somewhere in the **60–85%**
> range.

**JUST RUN this cell over and over**: each run deals a new headline.
Guess the outlet in your head before reading the reveal. Keep your own score against
the model's.

In [ ]:
# JUST RUN. Again and again.
quiz()

> **What you should see:** a headline, the model's guess, the actual outlet, and
> whether the model was right.

After ten rounds or so: what tells are you using to determine which headline is which? How do you think the model does?

**Edit** feed it made-up or real headlines and watch its confidence across the different outlets

In [ ]:
my_headline = "Markets furious as trade negotiators meet in Beijing"   # <-- EDIT

guess_outlet(my_headline)

> **What you should see:** your headline followed by four confidence bars, largest
> on top for the starting headline, WSJ should lead comfortably.

### Things to try

1. Change the wording to find
   the single word swap that moves the most probability.
2. Write a headline about something with no US-China content at all: a bake sale, a
   football game, World Cup. The model *still* picks an outlet because it has to. What does that tell you about trusting a classifier outside the world it
   was trained on?

---
## Fool it

Write headlines that carry one
outlet's "worldview" in another outlet's "style". A pro-Beijing message in WSJ
market-talk. A Fox-style emotional rhetoric written with NYT talk.

**Edit** replace these with your own attempts, then run.

In [ ]:
my_headlines = [
    "Investors cheer as tariffs teach Beijing a lesson",          # <-- EDIT
    "Analysts question whether balloon response was warranted",    # <-- EDIT
    "Win-win cooperation lifts stock futures higher",              # <-- EDIT: add more, comma-separated
]

battery(my_headlines)

> **What you should see:** a table pairing each of your headlines with the model's
> guess.

### Things to try

1. Score yourself as a successful fool is a headline where you intended one outlet's
   worldview but the model names another. What's your hit rate after ten attempts?
2. Reverse-engineer your wins so when you fooled it, which words did the work?  This is a real research technique called probing in order to determine what the model is picking up / learned.
3. A bigger question: when the model says "Fox," is it detecting ideology, or
   just statistics on the vocab the dataset has that happens to relate with ideology? Which side is it? Or in other words is the model processing language or understanding it?

---
## Style or subject?

So do you belive the model can or can't learn
outlet style? Did it just memorize which words go with which events?

To test we can train it with an entire event hidden from it's training dataset, then test it only on
that unseen event. If it still guesses outlets well, it learned style that
transfers, but if it collapses, it was leaning on event vocabulary.

**Edit:**

In [ ]:
hidden_event = "Spy balloon"   # <-- EDIT (any event from the list)

train_rows = headlines[headlines.event != hidden_event]
test_rows  = headlines[headlines.event == hidden_event]
m2 = LogisticRegression(max_iter=2000)
m2.fit(embedder.encode(list(train_rows.headline)), train_rows.outlet)
acc = m2.score(embedder.encode(list(test_rows.headline)), test_rows.outlet)
print(f"Trained with '{hidden_event}' completely hidden.")
print(f"Accuracy on that unseen event: {acc:.0%}  (random = 25%)")

> **What you should see:** an accuracy number for the hidden event. Above ~50% means it generally did detect style, but near 25% means the model was memorizing event words.

Rotate through all six events. Is transfer even, or are some events easier to
generalize to?

Would you ever trust this model in real world applications? Why or why not?

In here we actually get to see how well it's accuracy is, but how well do you think the scores are for models out there running currently?

Does this make you trust or not trust trained models from language, image detection, audio detection, and to even smell detection?

---
## Recap

With edited strings and lists in Python in a Jupyter Notebook:

- measured how four outlets spin the same events
- found VADER's (the model) blind spots
- trained a classifier that identifies an outlet from wording alone,
- tested it with adversarial headlines, and
- ran a held-out domain test to ask whether it learned style or memorized vocabulary.

For technical speak, this work relates to Natural Language Processing (NLP) which is a catergory of models like ChatGPT and Claude (Large Language Modelss/LLMs) where the goal is too process language. However note that the model we use is a much simplier model that utilizes sentiment analysis to determine the tone or "emotion" of a text from a scale of -1 to 1.


In the world of LLMs, some will say that these new models now (Sol, Fable, etc) are getting closer to NLU (Natural Language Understanding) instead of NLP. Do you believe them? Do you think we will ever develop an NLU model?

Lastly any thoughts, ideas, concerns on this assignment as a whole?